In [ ]:
pip install opencv-python numpy pytesseract pdf2image pandas nltk symspellpy tqdm

In [ ]:
# Import required libraries
import cv2
import numpy as np
import pytesseract
import pandas as pd
import logging
import re
import nltk
from pdf2image import convert_from_path
from tqdm import tqdm
from symspellpy import SymSpell, Verbosity  # Fast OCR error correction

# Download necessary NLTK data
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

# Configure logging
logging.basicConfig(level=logging.INFO)

# Initialize SymSpell for automatic OCR correction
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
sym_spell.load_dictionary("frequency_dictionary_en_82_765.txt", term_index=0, count_index=1)

# OCR Configuration
TESSERACT_CONFIG = r'--oem 3 --psm 3'

In [ ]:
### Convert PDF to images with higher DPI
def pdf_to_images(pdf_path, dpi=400):
    logging.info(f"Converting PDF to images at {dpi} DPI...")
    images = convert_from_path(pdf_path, dpi=dpi, poppler_path="/opt/homebrew/bin/")
    logging.info(f"Extracted {len(images)} pages from PDF.")
    return images

### Preprocess images before OCR
def preprocess_image(image):
    logging.info("Preprocessing image for OCR...")

    # Convert to grayscale
    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)

    # Adaptive Thresholding for better OCR readability
    binary = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
    )

    # Deskew image to prevent text misalignment
    coords = np.column_stack(np.where(binary > 0))
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    (h, w) = binary.shape[:2]
    center = (w // 2, h // 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    deskewed = cv2.warpAffine(binary, rotation_matrix, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

    return deskewed

### Extract text from images using OCR
def extract_text_from_image(image):
    logging.info("Extracting text using OCR...")
    text = pytesseract.image_to_string(image, config=TESSERACT_CONFIG)
    return text.strip()

### Process OCR on entire PDF
def process_ocr(pdf_path):
    images = pdf_to_images(pdf_path)

    extracted_texts = []
    for img in tqdm(images, desc="Processing Pages"):
        preprocessed_img = preprocess_image(img)
        extracted_texts.append(extract_text_from_image(preprocessed_img))

    extracted_text = "\n".join(extracted_texts)
    logging.info("Finished OCR processing.")
    return extracted_text

### Correct OCR errors using SymSpell
def correct_ocr_text(text):
    words = text.split()
    corrected_words = [
        sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)[0].term if sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2) else word
        for word in words
    ]
    return " ".join(corrected_words)

### Text Cleaning and Formatting
def clean_text(text):
    logging.info("Cleaning extracted text...")

    # Step 1: Fix OCR misinterpretations using SymSpell
    text = correct_ocr_text(text)

    # Step 2: Preserve numbers, dates, and currency values
    text = re.sub(r'(\d+)\s+([a-zA-Z])', r'\1\2', text)  # Fix numbers breaking from words
    text = re.sub(r'([a-zA-Z])\s+(\d+)', r'\1\2', text)  # Fix words breaking from numbers
    text = re.sub(r'(?<=\d)[,.](?=\d)', '', text)  # Remove misinterpreted commas in numbers

    # Step 3: Standardize spaces and special characters
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII characters
    text = re.sub(r'\s+', ' ', text).strip()  # Reduce multiple spaces

    # Step 4: Tokenize text into sentences
    sentences = nltk.sent_tokenize(text)
    
    # Step 5: Remove stopwords and apply lemmatization
    stop_words = set(nltk.corpus.stopwords.words('english'))
    lemmatizer = nltk.WordNetLemmatizer()
    
    final_sentences = []
    for sentence in sentences:
        tokens = nltk.word_tokenize(sentence.lower())
        tokens = [word for word in tokens if word.isalnum()]  # Remove special characters
        tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords
        tokens = [lemmatizer.lemmatize(word) for word in tokens]  # Lemmatization
        final_sentences.append(" ".join(tokens))

    cleaned_text = ". ".join(final_sentences)  # Join cleaned sentences with periods
    logging.info(f"Cleaned text length: {len(cleaned_text)} characters.")
    return cleaned_text

### Process PDF and save cleaned data
def process_pdf(pdf_path, output_csv="Cleaned_Text.csv"):
    extracted_text = process_ocr(pdf_path)
    
    # Print a sample of extracted text
    logging.info(f"Sample Extracted Text:\n{extracted_text[:500]}")
    
    cleaned_data = clean_text(extracted_text)
    
    # Save cleaned data to CSV
    df = pd.DataFrame({"text": [cleaned_data]})
    df.to_csv(output_csv, index=False)
    logging.info(f"Cleaned data saved to {output_csv}")
    
    logging.info("Finished processing PDF.")
    return cleaned_data

### Run the process
pdf_path = "UHN-Sustainability-Report.pdf"  # Replace with actual path
cleaned_data = process_pdf(pdf_path)

### Print cleaned data sample
print(cleaned_data[:1000])  # Print first 1000 characters